# AI-Powered Loan Approval & Risk Intelligence Dashboard

Project code for the internship submission.

**Name:** Samruddhi Misal

**Dataset:** Loan-Approval-Prediction-Dataset (Kaggle)

## Project Objective

The main aim of this project is to study loan application data, find useful patterns, and present the results through an interactive dashboard. It also includes a simple machine-learning model for an approval prediction.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

st.set_page_config(
    page_title="Loan Approval & Risk Intelligence",
    page_icon="🏦",
    layout="wide"
)

DATA_FILE = "loan_approval_dataset.csv"

@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE)
    # The supplied dataset has leading spaces in most column names.
    df.columns = df.columns.str.strip()
    return df

@st.cache_resource
def train_model(df):
    target = "loan_status"
    features = [
        "no_of_dependents", "education", "self_employed",
        "income_annum", "loan_amount", "loan_term", "cibil_score",
        "residential_assets_value", "commercial_assets_value",
        "luxury_assets_value", "bank_asset_value"
    ]

    X = df[features].copy()
    y = df[target].map({"Approved": 1, "Rejected": 0})

    categorical = ["education", "self_employed"]
    numeric = [c for c in features if c not in categorical]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical)
        ]
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "confusion": confusion_matrix(y_test, pred)
    }
    return model, metrics

def money(x):
    return f"₹{x/1_000_000:.2f}M"

def build_insights(df):
    approved = df[df["loan_status"] == "Approved"]
    rejected = df[df["loan_status"] == "Rejected"]

    approval_rate = (df["loan_status"].eq("Approved").mean()) * 100

    cibil_bins = pd.cut(
        df["cibil_score"],
        bins=[0, 550, 650, 750, 900],
        labels=["<550", "550–649", "650–749", "750+"],
        include_lowest=True
    )
    cibil_rates = (
        df.assign(CIBIL_Band=cibil_bins)
        .groupby("CIBIL_Band", observed=False)["loan_status"]
        .apply(lambda s: (s == "Approved").mean() * 100)
        .reset_index(name="Approval_Rate")
    )

    best_cibil = cibil_rates.loc[cibil_rates["Approval_Rate"].idxmax()]
    worst_cibil = cibil_rates.loc[cibil_rates["Approval_Rate"].idxmin()]

    emp_rates = (
        df.groupby("self_employed")["loan_status"]
        .apply(lambda s: (s == "Approved").mean() * 100)
        .reset_index(name="Approval_Rate")
    )
    best_emp = emp_rates.loc[emp_rates["Approval_Rate"].idxmax()]

    return {
        "approval_rate": approval_rate,
        "avg_approved_loan": approved["loan_amount"].mean(),
        "avg_rejected_loan": rejected["loan_amount"].mean(),
        "best_cibil": best_cibil,
        "worst_cibil": worst_cibil,
        "best_emp": best_emp,
        "cibil_rates": cibil_rates,
        "emp_rates": emp_rates,
    }

df = load_data()
model, metrics = train_model(df)
ins = build_insights(df)

# ---------- Sidebar ----------
st.sidebar.title("🏦 Loan Intelligence")
st.sidebar.caption("AI-Powered Loan Approval & Risk Intelligence")
page = st.sidebar.radio(
    "Navigate",
    ["Executive Overview", "Applicant Intelligence", "Financial Risk", "Decision Intelligence", "Loan Predictor"]
)

st.sidebar.divider()
st.sidebar.info(
    "Project focus: convert loan application data into measurable insights, "
    "risk signals and practical decision support."
)

# ---------- Header ----------
st.title("🏦 Loan Approval & Risk Intelligence")
st.caption("Data → Insight → Risk/Opportunity → Action")

if page == "Executive Overview":
    st.subheader("Executive Overview")

    total = len(df)
    approved = int((df["loan_status"] == "Approved").sum())
    rejected = total - approved

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Total Applications", f"{total:,}")
    c2.metric("Approved", f"{approved:,}")
    c3.metric("Approval Rate", f"{ins['approval_rate']:.1f}%")
    c4.metric("Avg Loan Amount", money(df["loan_amount"].mean()))
    c5.metric("Avg CIBIL Score", f"{df['cibil_score'].mean():.0f}")

    st.divider()

    left, right = st.columns(2)
    with left:
        status = df["loan_status"].value_counts().reset_index()
        status.columns = ["Loan Status", "Applications"]
        fig = px.pie(
            status, names="Loan Status", values="Applications",
            hole=0.55, title="Approval Distribution"
        )
        st.plotly_chart(fig, use_container_width=True)

    with right:
        cibil = df.copy()
        cibil["CIBIL Band"] = pd.cut(
            cibil["cibil_score"],
            bins=[0, 550, 650, 750, 900],
            labels=["<550", "550–649", "650–749", "750+"],
            include_lowest=True
        )
        fig = px.bar(
            cibil.groupby("CIBIL Band", observed=False)["loan_status"]
            .apply(lambda x: (x == "Approved").mean() * 100)
            .reset_index(name="Approval Rate"),
            x="CIBIL Band", y="Approval Rate",
            title="Approval Rate by CIBIL Band",
            text_auto=".1f"
        )
        fig.update_yaxes(range=[0, 100], ticksuffix="%")
        st.plotly_chart(fig, use_container_width=True)

    st.subheader("Executive Takeaways")
    st.success(
        f"**Approval rate:** {ins['approval_rate']:.1f}% of applications were approved. "
        f"**CIBIL signal:** the highest observed approval rate occurs in the "
        f"{ins['best_cibil']['CIBIL_Band']} band ({ins['best_cibil']['Approval_Rate']:.1f}%). "
        f"**Attention area:** the lowest observed rate is in {ins['worst_cibil']['CIBIL_Band']} "
        f"({ins['worst_cibil']['Approval_Rate']:.1f}%)."
    )

elif page == "Applicant Intelligence":
    st.subheader("👤 Applicant Intelligence")

    col1, col2 = st.columns(2)

    with col1:
        edu = (
            df.groupby("education")["loan_status"]
            .apply(lambda x: (x == "Approved").mean() * 100)
            .reset_index(name="Approval Rate")
        )
        fig = px.bar(
            edu, x="education", y="Approval Rate",
            title="Approval Rate by Education",
            text_auto=".1f"
        )
        fig.update_yaxes(range=[0, 100], ticksuffix="%")
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        emp = (
            df.groupby("self_employed")["loan_status"]
            .apply(lambda x: (x == "Approved").mean() * 100)
            .reset_index(name="Approval Rate")
        )
        fig = px.bar(
            emp, x="self_employed", y="Approval Rate",
            title="Approval Rate by Employment Type",
            text_auto=".1f"
        )
        fig.update_yaxes(range=[0, 100], ticksuffix="%")
        st.plotly_chart(fig, use_container_width=True)

    fig = px.scatter(
        df.sample(min(1500, len(df)), random_state=42),
        x="income_annum", y="loan_amount",
        color="loan_status",
        size="cibil_score",
        hover_data=["education", "self_employed", "loan_term"],
        title="Income vs Loan Amount"
    )
    st.plotly_chart(fig, use_container_width=True)

    st.info(
        "Interpretation note: these charts show associations in the observed dataset. "
        "They do not by themselves prove that education, employment or income causes approval."
    )

elif page == "Financial Risk":
    st.subheader("⚠️ Financial Risk Analysis")

    data = df.copy()
    data["Loan_to_Income"] = data["loan_amount"] / data["income_annum"].replace(0, np.nan)
    data["Total_Assets"] = (
        data["residential_assets_value"]
        + data["commercial_assets_value"]
        + data["luxury_assets_value"]
        + data["bank_asset_value"]
    )
    data["Asset_to_Loan"] = data["Total_Assets"] / data["loan_amount"].replace(0, np.nan)

    c1, c2, c3 = st.columns(3)
    c1.metric("Median Loan / Income", f"{data['Loan_to_Income'].median():.2f}×")
    c2.metric("Median Total Assets", money(data["Total_Assets"].median()))
    c3.metric("Median CIBIL", f"{data['cibil_score'].median():.0f}")

    left, right = st.columns(2)
    with left:
        fig = px.box(
            data, x="loan_status", y="Loan_to_Income",
            color="loan_status",
            title="Loan-to-Income Ratio by Decision"
        )
        st.plotly_chart(fig, use_container_width=True)

    with right:
        fig = px.scatter(
            data.sample(min(1500, len(data)), random_state=42),
            x="cibil_score", y="loan_amount",
            color="loan_status",
            size="income_annum",
            title="CIBIL Score vs Loan Amount"
        )
        st.plotly_chart(fig, use_container_width=True)

    risk = data.copy()
    risk["Risk Flag"] = np.select(
        [
            (risk["cibil_score"] < 550) & (risk["Loan_to_Income"] > 4),
            (risk["cibil_score"] < 650) | (risk["Loan_to_Income"] > 5)
        ],
        ["High attention", "Review"],
        default="Lower attention"
    )

    st.subheader("Illustrative Risk Segmentation")
    counts = risk["Risk Flag"].value_counts().reset_index()
    counts.columns = ["Risk Flag", "Applications"]
    fig = px.bar(
        counts, x="Risk Flag", y="Applications",
        title="Applications by Risk Flag",
        text_auto=True
    )
    st.plotly_chart(fig, use_container_width=True)

elif page == "Decision Intelligence":
    st.subheader("🧠 Decision Intelligence")

    st.markdown("### FACT")
    st.write(
        f"The dataset contains **{len(df):,} applications**, of which "
        f"**{(df['loan_status'] == 'Approved').sum():,}** were approved. "
        f"The overall observed approval rate is **{ins['approval_rate']:.1f}%**."
    )

    st.markdown("### INSIGHT")
    st.write(
        f"The strongest approval-rate difference visible in the dashboard is across "
        f"CIBIL bands. The **{ins['best_cibil']['CIBIL_Band']}** band has an observed "
        f"approval rate of **{ins['best_cibil']['Approval_Rate']:.1f}%**, while "
        f"**{ins['worst_cibil']['CIBIL_Band']}** has **{ins['worst_cibil']['Approval_Rate']:.1f}%**."
    )

    st.markdown("### ⚠️ RISK")
    st.warning(
        "Applications combining weaker credit scores with a relatively high "
        "loan-to-income ratio deserve additional review. This is a screening signal, "
        "not proof of applicant default risk."
    )

    st.markdown("### 💡 OPPORTUNITY")
    st.success(
        "Use transparent risk segmentation to prioritize manual review and reduce "
        "unnecessary review effort for applications with stronger observed profiles."
    )

    st.markdown("### ACTION")
    st.write(
        "1. Use CIBIL and loan-to-income bands as review-support indicators.  \n"
        "2. Monitor approval rates by applicant segment regularly.  \n"
        "3. Investigate borderline applications separately rather than applying one rule to everyone.  \n"
        "4. Re-train and validate the model periodically if it is used in production."
    )

    st.divider()
    st.subheader("Model Validation Snapshot")
    m1, m2, m3 = st.columns(3)
    m1.metric("Accuracy", f"{metrics['accuracy']*100:.1f}%")
    m2.metric("Precision", f"{metrics['precision']*100:.1f}%")
    m3.metric("Recall", f"{metrics['recall']*100:.1f}%")

    st.caption(
        "The model is a demonstration for this project, not a production credit-decision system."
    )

elif page == "Loan Predictor":
    st.subheader("🔮 Loan Approval Predictor")
    st.write("Enter applicant details to generate a model-based approval estimate.")

    a, b, c = st.columns(3)
    with a:
        dependents = st.number_input("Dependents", 0, 10, 2)
        education = st.selectbox("Education", ["Graduate", "Not Graduate"])
        self_employed = st.selectbox("Self Employed", ["No", "Yes"])
        income = st.number_input("Annual Income (₹)", min_value=100000, value=5000000, step=100000)

    with b:
        loan_amount = st.number_input("Loan Amount (₹)", min_value=100000, value=15000000, step=100000)
        loan_term = st.number_input("Loan Term (years)", min_value=1, max_value=30, value=10)
        cibil = st.number_input("CIBIL Score", min_value=300, max_value=900, value=700)

    with c:
        residential = st.number_input("Residential Assets (₹)", min_value=0, value=5000000, step=100000)
        commercial = st.number_input("Commercial Assets (₹)", min_value=0, value=5000000, step=100000)
        luxury = st.number_input("Luxury Assets (₹)", min_value=0, value=10000000, step=100000)
        bank = st.number_input("Bank Assets (₹)", min_value=0, value=5000000, step=100000)

    if st.button("Predict Approval", type="primary", use_container_width=True):
        input_df = pd.DataFrame([{
            "no_of_dependents": dependents,
            "education": education,
            "self_employed": self_employed,
            "income_annum": income,
            "loan_amount": loan_amount,
            "loan_term": loan_term,
            "cibil_score": cibil,
            "residential_assets_value": residential,
            "commercial_assets_value": commercial,
            "luxury_assets_value": luxury,
            "bank_asset_value": bank
        }])

        probability = model.predict_proba(input_df)[0, 1]
        prediction = "Approved" if probability >= 0.5 else "Rejected"

        if prediction == "Approved":
            st.success(f"### Predicted: {prediction}")
        else:
            st.error(f"### Predicted: {prediction}")

        st.metric("Approval Probability", f"{probability*100:.1f}%")

        signals = []
        if cibil < 650:
            signals.append("CIBIL score is below 650.")
        if income > 0 and loan_amount / income > 5:
            signals.append("Loan-to-income ratio is relatively high.")
        if loan_term >= 20:
            signals.append("Long loan term may require additional review.")
        if not signals:
            signals.append("No major screening signal was triggered by the simple project rules.")

        st.subheader("Screening Signals")
        for s in signals:
            st.write("• " + s)

        st.caption(
            "This prediction is based on a simple logistic-regression demonstration trained "
            "on the supplied dataset. It should not be used as a real lending decision."
        )
